# DiamondsCount

[Codility task](https://app.codility.com/programmers/trainings/3/diamonds_count/) - Exercise 3, 2017 Contest

A *diamond* has four equal sides and diagonals parallel to the coordinate axes. `solution(X, Y)` counts the diamonds whose four vertices are among the `N` given distinct points (different vertex sets count as different diamonds; empty-area diamonds do not count).

| X | Y | result |
|---|---|--------|
| `[1, 1, 2, 2, 2, 3, 3]` | `[3, 4, 1, 3, 5, 3, 4]` | 2 |
| `[1, 2, 3, 3, 2, 1]` | `[1, 1, 1, 2, 2, 2]` | 0 |

Assumptions: `N` in `[4..1,500]`, coordinates in `[0..N-1]`, points pairwise distinct.

## Idea

A quadrilateral with equal sides is a rhombus, and its diagonals bisect each other. With axis-parallel diagonals the shape is:

- a horizontal diagonal `(cx - a, cy) - (cx + a, cy)`,
- a vertical diagonal `(cx, cy - b) - (cx, cy + b)`, with `a, b > 0`,

both with the same midpoint `(cx, cy)`. Because the horizontal endpoints have integer `y` and the vertical ones integer `x`, the centre is an **integer point** (it does not need to be one of the given points). A diamond is determined by its centre and its two diagonals, independent choices:

`diamonds = sum over centres of h(centre) * v(centre)`

- `h` = number of pairs of points on row `cy` symmetric about `cx`;
- `v` = number of pairs of points on column `cx` symmetric about `cy`.

Enumerating the pairs inside each row costs `sum k(k-1)/2 <= N(N-1)/2`, the same for the columns: **O(N^2)** time. A dictionary stores `h`, then each column pair simply looks its centre up.

In [ ]:
from collections import defaultdict


def solution(X, Y):
    base = max(max(X), max(Y)) + 1                  # to pack (x, y) into one integer
    rows = defaultdict(list)
    cols = defaultdict(list)
    for x, y in zip(X, Y):
        rows[y].append(x)
        cols[x].append(y)

    horizontal = defaultdict(int)                   # centre -> pairs on that row symmetric about it
    for y, xs in rows.items():
        for i in range(len(xs)):
            for j in range(i + 1, len(xs)):
                s = xs[i] + xs[j]
                if s % 2 == 0:
                    horizontal[(s // 2) * base + y] += 1

    if not horizontal:
        return 0
    total = 0
    for x, ys in cols.items():
        for i in range(len(ys)):
            for j in range(i + 1, len(ys)):
                s = ys[i] + ys[j]
                if s % 2 == 0:
                    total += horizontal.get(x * base + s // 2, 0)
    return total

## Official examples

In [ ]:
assert solution([1, 1, 2, 2, 2, 3, 3], [3, 4, 1, 3, 5, 3, 4]) == 2
assert solution([1, 2, 3, 3, 2, 1], [1, 1, 1, 2, 2, 2]) == 0
print("examples OK")

## Edge cases

In [ ]:
assert solution([0, 1, 2, 1], [1, 0, 1, 2]) == 1                   # a square rotated 45 degrees; its centre (1, 1) is not a given point
assert solution([0, 1, 2, 1, 1], [1, 0, 1, 2, 1]) == 1             # the same plus its centre: still one diamond
assert solution([0, 4, 2, 2], [2, 2, 1, 3]) == 1                   # diagonals of different lengths (4 and 2)
assert solution([0, 1, 0, 1], [0, 0, 1, 1]) == 0                   # a square with axis-parallel SIDES: its diagonals are not axis-parallel
assert solution([0, 1, 2, 3], [0, 0, 0, 0]) == 0                   # collinear points
assert solution([0, 2, 1, 1], [1, 1, 0, 3]) == 0                   # the diagonals have different midpoints: not a diamond
assert solution([0, 3, 1, 1], [0, 0, 1, 3]) == 0                   # (0,0)-(3,0) has no integer centre
# two row pairs and two column pairs around the same centre (2, 2): 2 * 2 diamonds
assert solution([0, 1, 3, 4, 2, 2, 2, 2], [2, 2, 2, 2, 0, 1, 3, 4]) == 4
print("edge cases OK")

## Brute force

The reference checks **every set of four points** with the geometric definition: some ordering `p0 p1 p2 p3` has four equal sides (squared distances), non-zero area (shoelace formula) and both diagonals (`p0 p2` and `p1 p3`) parallel to an axis. It is used with small random point sets on a small grid, so that diamonds are frequent.

In [ ]:
import itertools
import random


def d2(p, q):
    return (p[0] - q[0]) ** 2 + (p[1] - q[1]) ** 2


def is_diamond(points):
    p0 = points[0]
    for p1, p2, p3 in itertools.permutations(points[1:]):
        sides = {d2(p0, p1), d2(p1, p2), d2(p2, p3), d2(p3, p0)}
        if len(sides) != 1:
            continue
        axis_parallel = (p0[0] == p2[0] or p0[1] == p2[1]) and (p1[0] == p3[0] or p1[1] == p3[1])
        area2 = sum(a[0] * b[1] - b[0] * a[1] for a, b in [(p0, p1), (p1, p2), (p2, p3), (p3, p0)])
        if axis_parallel and area2 != 0:
            return True
    return False


def brute(X, Y):
    pts = list(zip(X, Y))
    return sum(1 for quad in itertools.combinations(pts, 4) if is_diamond(quad))


rng = random.Random(5)
nonzero = 0
for _ in range(1500):
    k = rng.randint(3, 5)                                          # points live on a k x k grid
    n = rng.randint(4, min(12, k * k))
    pts = rng.sample([(x, y) for x in range(k) for y in range(k)], n)
    X = [p[0] for p in pts]
    Y = [p[1] for p in pts]
    expected = brute(X, Y)
    assert solution(X, Y) == expected, (X, Y)
    nonzero += expected > 0
print(f"matches brute force on 1500 random point sets ({nonzero} contain at least one diamond)")

## Closed formula: full grids

In a full `g x g` grid every integer centre `(i, j)` has `min(i, g-1-i)` row pairs and `min(j, g-1-j)` column pairs, so the answer is `(sum_i min(i, g-1-i))^2`. This checks bigger inputs than the brute force can.

In [ ]:
def grid(g):
    X = [x for x in range(g) for y in range(g)]
    Y = [y for x in range(g) for y in range(g)]
    return X, Y


for g in range(2, 31):
    expected = sum(min(i, g - 1 - i) for i in range(g)) ** 2
    assert solution(*grid(g)) == expected, g
print("full grids g x g for g = 2..30 match the closed formula")

## Performance (N = 1,500)

In [ ]:
import time


def timed(name, X, Y):
    assert len(X) <= 1500 and len(set(zip(X, Y))) == len(X)
    start = time.perf_counter()
    result = solution(X, Y)
    print(f"{name:32s} N={len(X):5d} -> {result:>12} in {time.perf_counter() - start:.3f}s")


timed("one full row", list(range(1500)), [0] * 1500)
timed("row + column crossing", list(range(750)) + [375] * 749, [375] * 750 + [y for y in range(750) if y != 375][:749])

X, Y = grid(39)
timed("38 columns x 39 rows grid", X[:1482], Y[:1482])

pts = rng.sample([(x, y) for x in range(60) for y in range(60)], 1500)
timed("random on 60 x 60", [p[0] for p in pts], [p[1] for p in pts])

## Notes

- Counting by **centre** avoids looking at quadruples (O(N^4)) or triples (O(N^3)): the pairs on a line are enough, which is O(N^2).
- The two diagonals are independent given the centre, hence the product `h * v`.
- Parity check: points have integer coordinates, so a pair is symmetric about an integer centre only when the sum of its coordinates is even.
- The brute force confirms the geometric reasoning; the full grids confirm it at scale with an exact formula.